1 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import re
import string

from tqdm import tqdm

2 — Load Dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
df = pd.read_csv("dataset_balanced_C0-C4.csv")
df.head()

Saving dataset_balanced_C0-C4.csv to dataset_balanced_C0-C4 (1).csv


,No,Komentar,Label Awal,Label Pakar,Pasal Dominan,Catatan,Unnamed: 6,Kode Label,jumlah_kata,jumlah_karakter,clean_comment
0,5708.0,Apakah seorang maling anak'y jg akan menjadi m...,Indikasi penghinaan atau kebencian terhadap in...,Ambigu atau konteks tidak cukup,NaN,Pertanyaan umum tanpa target yang jelas.,NaN,C4,8.0,56.0,apakah seorang maling anak'y jg akan menjadi m...
1,495.0,"​@HanaFanda-wx1cciyaa betul,\nMeni hese ngesah...",Netral atau tidak terindikasi,Konten ofensif,NaN,Tidak terindikasi Pasal 27A maupun Pasal 28 ay...,NaN,C2,12.0,91.0,"​@hanafanda-wx1cciyaa betul, meni hese ngesahk..."
2,1841.0,Omongan nya pak bahlil sering kontradiksi. Lah...,Netral atau tidak terindikasi,Netral atau tidak terindikasi,NaN,NaN,NaN,C0,7.0,50.0,omongan nya pak bahlil sering kontradiksi. lah...
3,6483.0,Astagfirullah. Katanya Wakil Rakyat tapi kelak...,Indikasi penghinaan atau kebencian terhadap in...,Konten ofensif,NaN,Menuduh wakil rakyat sebagai maling dan zalim.,NaN,C2,13.0,96.0,astagfirullah. katanya wakil rakyat tapi kelak...
4,2222.0,"Cobalah pampang ke45 poin itu di media sosial,...",Netral atau tidak terindikasi,Ambigu atau konteks tidak cukup,NaN,komentar ini belum menunjukkan unsur yang cuku...,NaN,C4,12.0,76.0,"cobalah pampang ke45 poin itu di media sosial,..."


3 — Struktur Dataset

In [ ]:
df.info()

print("Jumlah data :", len(df))
df['Kode Label'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3815 entries, 0 to 3814
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   No               3597 non-null   float64
 1   Komentar         3815 non-null   object 
 2   Label Awal       3815 non-null   object 
 3   Label Pakar      3815 non-null   object 
 4   Pasal Dominan    637 non-null    object 
 5   Catatan          3384 non-null   object 
 6   Unnamed: 6       1 non-null      object 
 7   Kode Label       3815 non-null   object 
 8   jumlah_kata      3597 non-null   float64
 9   jumlah_karakter  3597 non-null   float64
 10  clean_comment    3815 non-null   object 
dtypes: float64(3), object(8)
memory usage: 328.0+ KB
Jumlah data : 3815


,count
Kode Label,
C4,763
C2,763
C0,763
C1,763
C3,763


4 — Contoh Komentar Awal

In [ ]:
df[["Komentar","Kode Label"]].sample(10,random_state=42)

,Komentar,Kode Label
1078,ya betul buk suruh buktikanan mana PKInya........,C2
1957,TUHAN TDK AKAN MEMBERKATI DLM KEBURUKAN APLG K...,C0
2694,"@fatcrruise6508 klo pengen jadi pahlawan, p...",C2
2002,UU perampasan aset para koruptor harus segera ...,C0
1213,Iya Bu....di negri ini banyak yg meruksak...co...,C1
1104,Gugat Gribran wapres plonga plongo . Ini naman...,C3
2654,Keliatan dari ruang kerja yang banyak mainan w...,C2
1694,Memberi jalan kepada komunis untuk berkembang ...,C0
721,Parcok saat ini tempat berkumpulnya para mafia...,C2
949,Selamat berjuang...membrantas mafia di KPK,C1


5 — Memuat Kamus Normalisasi Slang

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json

file_mapping = "/content/drive/MyDrive/Bimbingan_27-08/full_mapping_slang_corpus_final.json"
with open(file_mapping,"r",encoding="utf-8") as f:full_mapping_slang_corpus_final = json.load(f)
print("Jumlah slang mapping:",len(full_mapping_slang_corpus_final))

Jumlah slang mapping: 5042


In [ ]:
list(full_mapping_slang_corpus_final.items())[:10]

[('merekak', 'mereka'),
 ('emng', 'memang'),
 ('ngelewati', 'melewati'),
 ('onoh', 'sono'),
 ('tutorialx', 'tutorialnya'),
 ('mmperbaiki', 'memperbaiki'),
 ('prsatukn', 'persatukan'),
 ('gasih', 'enggak sih'),
 ('erempuan', 'perempuan'),
 ('klmpk', 'kelompok')]

6 — Fungsi Cleaning Komentar

In [ ]:
def cleaning_text(text):
    text = str(text)
    # case folding
    text = text.lower()

    # hapus URL
    text = re.sub(r"http\S+|www\S+","",text)

    # hapus mention
    text = re.sub(r"@\w+","",text)

    # normalisasi karakter berulang
    text = re.sub(r"(.)\1{2,}",r"\1",text)

    # hapus whitespace berlebih
    text = re.sub(r"\s+"," ",text)

    return text.strip()

7 — Fungsi Normalisasi Slang

In [ ]:
def normalize_slang(text,mapping):
    words = text.split()
    normalized_words = []

    for word in words:
        normalized_words.append(
            mapping.get(word,word))

    return " ".join(normalized_words)

8 — Cleaning

In [ ]:
df["Komentar_clean"] = (df["Komentar"].apply(cleaning_text))

In [ ]:
df[["Komentar","Komentar_clean"]].head()

,Komentar,Komentar_clean
0,Apakah seorang maling anak'y jg akan menjadi m...,apakah seorang maling anak'y jg akan menjadi m...
1,"​@HanaFanda-wx1cciyaa betul,\nMeni hese ngesah...","​-wx1cciyaa betul, meni hese ngesahkeun pearam..."
2,Omongan nya pak bahlil sering kontradiksi. Lah...,omongan nya pak bahlil sering kontradiksi. lah...
3,Astagfirullah. Katanya Wakil Rakyat tapi kelak...,astagfirullah. katanya wakil rakyat tapi kelak...
4,"Cobalah pampang ke45 poin itu di media sosial,...","cobalah pampang ke45 poin itu di media sosial,..."


9 — Normalisasi Slang

In [ ]:
df["Komentar_preprocessed"] = (
    df["Komentar_clean"].apply(lambda x: normalize_slang(x, full_mapping_slang_corpus_final)))

10 — Hasil Preprocessing

In [ ]:
df[["Komentar","Komentar_preprocessed","Kode Label"]].sample(10,random_state=42)

,Komentar,Komentar_preprocessed,Kode Label
1078,ya betul buk suruh buktikanan mana PKInya........,ya betul buk suruh buktikanan mana pkinya.mang...,C2
1957,TUHAN TDK AKAN MEMBERKATI DLM KEBURUKAN APLG K...,tuhan tidak akan memberkati dalam keburukan ap...,C0
2694,"@fatcrruise6508 klo pengen jadi pahlawan, p...","kalau ingin jadi pahlawan, paling kesalahan fa...",C2
2002,UU perampasan aset para koruptor harus segera ...,uu perampasan aset para koruptor harus segera ...,C0
1213,Iya Bu....di negri ini banyak yg meruksak...co...,iya bu.di negeri ini banyak yang meruksak.cont...,C1
1104,Gugat Gribran wapres plonga plongo . Ini naman...,gugat gribran wapres plonga plongo . ini naman...,C3
2654,Keliatan dari ruang kerja yang banyak mainan w...,kelihatan dari ruang kerja yang banyak mainan ...,C2
1694,Memberi jalan kepada komunis untuk berkembang ...,memberi jalan kepada komunis untuk berkembang ...,C0
721,Parcok saat ini tempat berkumpulnya para mafia...,parcok saat ini tempat berkumpulnya para mafia...,C2
949,Selamat berjuang...membrantas mafia di KPK,selamat berjuang.membrantas mafia di kpk,C1


11 — Validasi Hasil Preprocessing

In [ ]:
df["Komentar_preprocessed"].isnull().sum()

np.int64(0)

In [ ]:
(df["Komentar_preprocessed"] == "").sum()

np.int64(0)

12 — Statistik Dataset Akhir

In [ ]:
print("Jumlah data akhir:",len(df))
print(df["Kode Label"].value_counts())

Jumlah data akhir: 3815
Kode Label
C4    763
C2    763
C0    763
C1    763
C3    763
Name: count, dtype: int64


13 — Cek Duplikasi Setelah Preprocessing

In [ ]:
df["Komentar_preprocessed"].duplicated().sum()

np.int64(9)

In [ ]:
df[df["Komentar_preprocessed"].duplicated(keep=False)
][["Komentar","Komentar_preprocessed","Kode Label"]]

,Komentar,Komentar_preprocessed,Kode Label
42,@sobatnu7078 INI ORANG DARI PARTAI SILIT IRE...,ini orang dari partai silit ireng sejatinya an...,C3
61,AAMIIN,amin,C0
271,Seeetuuuujuuuu,setuju,C0
424,"Lanjut atas, \nGusDur sendiri saat jadi presid...","lanjut atas, gusdur sendiri saat jadi presiden...",C3
523,INI ORANG DARI PARTAI SILIT IRENG SEJATINYA AN...,ini orang dari partai silit ireng sejatinya an...,C2
1027,setujuuu,setuju,C4
1194,@sitihajar7046 Astaga! 23 Warga Koja Keracun...,astaga! 23 warga koja keracunan usai santap na...,C2
1695,@himadunoim2529 Astaga! 23 Warga Koja Keracu...,astaga! 23 warga koja keracunan usai santap na...,C2
1740,@antipki5178 👍👍👍👍👍,👍,C0
2120,@AchmadSyaikhu INI ORANG DARI PARTAI SILIT I...,ini orang dari partai silit ireng sejatinya an...,C3


14 — Contoh Perubahan Slang

In [ ]:
df[["Komentar","Komentar_preprocessed"]].sample(20,random_state=42)

,Komentar,Komentar_preprocessed
1078,ya betul buk suruh buktikanan mana PKInya........,ya betul buk suruh buktikanan mana pkinya.mang...
1957,TUHAN TDK AKAN MEMBERKATI DLM KEBURUKAN APLG K...,tuhan tidak akan memberkati dalam keburukan ap...
2694,"@fatcrruise6508 klo pengen jadi pahlawan, p...","kalau ingin jadi pahlawan, paling kesalahan fa..."
2002,UU perampasan aset para koruptor harus segera ...,uu perampasan aset para koruptor harus segera ...
1213,Iya Bu....di negri ini banyak yg meruksak...co...,iya bu.di negeri ini banyak yang meruksak.cont...
1104,Gugat Gribran wapres plonga plongo . Ini naman...,gugat gribran wapres plonga plongo . ini naman...
2654,Keliatan dari ruang kerja yang banyak mainan w...,kelihatan dari ruang kerja yang banyak mainan ...
1694,Memberi jalan kepada komunis untuk berkembang ...,memberi jalan kepada komunis untuk berkembang ...
721,Parcok saat ini tempat berkumpulnya para mafia...,parcok saat ini tempat berkumpulnya para mafia...
949,Selamat berjuang...membrantas mafia di KPK,selamat berjuang.membrantas mafia di kpk


15 — Audit Duplikasi Berdasarkan Label

In [ ]:
df_duplikat = df[df["Komentar_preprocessed"].duplicated(keep=False)].copy()

audit_duplikat = (
    df_duplikat
    .groupby("Komentar_preprocessed")["Kode Label"]
    .agg(
        jumlah_data="size",
        jumlah_label="nunique",
        label=lambda x: ", ".join(sorted(x.unique())))
    .reset_index())

display(audit_duplikat)

,Komentar_preprocessed,jumlah_data,jumlah_label,label
0,amin,2,2,"C0, C4"
1,astaga! 23 warga koja keracunan usai santap na...,2,1,C2
2,ini orang dari partai silit ireng sejatinya an...,3,2,"C2, C3"
3,"lanjut atas, gusdur sendiri saat jadi presiden...",2,1,C3
4,setuju,3,2,"C0, C4"
5,👍,2,2,"C0, C4"
6,😊,2,1,C4


16 — Replacement Duplikasi

In [ ]:
from google.colab import files
import pandas as pd

# 1. IDENTIFIKASI KELOMPOK DUPLIKAT

mask_duplicate = (
    df["Komentar_preprocessed"]
    .duplicated(keep=False))

df_duplikat = df.loc[
    mask_duplicate,
    ["No", "Komentar", "Komentar_preprocessed", "Kode Label"]].copy()

print("Jumlah baris dalam kelompok duplicate:",len(df_duplikat))
print("\nKelompok duplicate:")
display(df_duplikat)

Jumlah baris dalam kelompok duplicate: 16

Kelompok duplicate:


,No,Komentar,Komentar_preprocessed,Kode Label
42,7211.0,@sobatnu7078 INI ORANG DARI PARTAI SILIT IRE...,ini orang dari partai silit ireng sejatinya an...,C3
61,1798.0,AAMIIN,amin,C0
271,7187.0,Seeetuuuujuuuu,setuju,C0
424,4959.0,"Lanjut atas, \nGusDur sendiri saat jadi presid...","lanjut atas, gusdur sendiri saat jadi presiden...",C3
523,1620.0,INI ORANG DARI PARTAI SILIT IRENG SEJATINYA AN...,ini orang dari partai silit ireng sejatinya an...,C2
1027,4955.0,setujuuu,setuju,C4
1194,2822.0,@sitihajar7046 Astaga! 23 Warga Koja Keracun...,astaga! 23 warga koja keracunan usai santap na...,C2
1695,3434.0,@himadunoim2529 Astaga! 23 Warga Koja Keracu...,astaga! 23 warga koja keracunan usai santap na...,C2
1740,1557.0,@antipki5178 👍👍👍👍👍,👍,C0
2120,7709.0,@AchmadSyaikhu INI ORANG DARI PARTAI SILIT I...,ini orang dari partai silit ireng sejatinya an...,C3


In [ ]:
# 16.1 — MENENTUKAN DATA C0-C2 YANG AKAN DIGANTI

index_replace = []
for nilai_duplicate, group in df_duplikat.groupby(
    "Komentar_preprocessed",
    sort=False):

    labels = group["Kode Label"].tolist()

    # Jika kelompok memiliki C3 atau C4, pertahankan C3/C4 dan ganti C0-C2
    if any(label in ["C3", "C4"] for label in labels):
        target = group[
            group["Kode Label"].isin(
                ["C0", "C1", "C2"])]

        index_replace.extend(target.index.tolist())

    # Jika kelompok hanya C0-C2, pertahankan kemunculan pertama, ganti sisanya
    else:
        index_keep = group.index[0]
        target = group.drop(index=index_keep)
        index_replace.extend(target.index.tolist())

print("Jumlah data C0-C2 yang akan di-replace:", len(index_replace))
print("\nData yang akan diganti:")

display(df.loc[index_replace,[
            "No",
            "Komentar",
            "Komentar_preprocessed",
            "Kode Label"]])

Jumlah data C0-C2 yang akan di-replace: 6

Data yang akan diganti:


,No,Komentar,Komentar_preprocessed,Kode Label
523,1620.0,INI ORANG DARI PARTAI SILIT IRENG SEJATINYA AN...,ini orang dari partai silit ireng sejatinya an...,C2
61,1798.0,AAMIIN,amin,C0
271,7187.0,Seeetuuuujuuuu,setuju,C0
2853,6800.0,Setujuuuu,setuju,C0
1695,3434.0,@himadunoim2529 Astaga! 23 Warga Koja Keracu...,astaga! 23 warga koja keracunan usai santap na...,C2
1740,1557.0,@antipki5178 👍👍👍👍👍,👍,C0


In [ ]:
# 16.2 — LOAD DATA ANOTASI PAKAR 8.000

from google.colab import files
import pandas as pd

uploaded = files.upload()

df_pakar = pd.read_excel("dataset_validasi_pakar.xlsx",sheet_name="Validasi Pakar")
df.head()

print("Jumlah data anotasi pakar:", len(df_pakar))
print("\nKolom data pakar:")
print(df_pakar.columns.tolist())

Saving dataset_validasi_pakar.xlsx to dataset_validasi_pakar (3).xlsx
Jumlah data anotasi pakar: 8000

Kolom data pakar:
['No', 'Komentar', 'Label Awal', 'Label Pakar', 'Pasal Dominan', 'Catatan', 'Unnamed: 6']


In [ ]:
# 16.3 — MENYIAPKAN KANDIDAT PENGGANTI

df_kandidat = df_pakar.copy()
df_kandidat["No"] = pd.to_numeric(df_kandidat["No"], errors="coerce")

# No yang sudah digunakan di dataset 3.815
no_digunakan = set(
    pd.to_numeric(
        df["No"],
        errors="coerce")
    .dropna()
    .astype(int))

# Komentar asli yang sudah digunakan
komentar_digunakan = set(
    df["Komentar"]
    .dropna()
    .astype(str))

# Ambil hanya C0-C2
label_mapping = {
    "Netral atau tidak terindikasi": "C0",
    "Kritik atau opini negatif": "C1",
    "Konten ofensif": "C2",
    "Indikasi penghinaan atau kebencian terhadap individu atau kelompok": "C3",
    "Ambigu atau konteks tidak cukup": "C4"}

df_kandidat["Kode Label"] = (
    df_kandidat["Label Pakar"]
    .map(label_mapping))

df_kandidat = df_kandidat[
    df_kandidat["Kode Label"].isin(
        ["C0", "C1", "C2"])].copy()

# Kandidat belum digunakan
df_kandidat = df_kandidat[
    ~df_kandidat["No"].isin(no_digunakan)].copy()

df_kandidat = df_kandidat[
    ~df_kandidat["Komentar"]
    .astype(str)
    .isin(komentar_digunakan)].copy()

# Buang komentar kosong
df_kandidat = df_kandidat[
    df_kandidat["Komentar"].notna()].copy()

print("Jumlah kandidat C0-C2:", len(df_kandidat))

Jumlah kandidat C0-C2: 4400


In [ ]:
# 16.4 — PREPROCESSING KANDIDAT

df_kandidat["Komentar_clean"] = (
    df_kandidat["Komentar"]
    .apply(cleaning_text))

df_kandidat["Komentar_preprocessed"] = (
    df_kandidat["Komentar_clean"]
    .apply(
        lambda x:
        normalize_slang(
            x,
            full_mapping_slang_corpus_final)))

print("Preprocessing kandidat selesai.")

Preprocessing kandidat selesai.


In [ ]:
# 16.5 — VALIDASI KANDIDAT

# Semua hasil preprocessing yang sudah ada
preprocessed_digunakan = set(
    df["Komentar_preprocessed"]
    .dropna()
    .astype(str))

# Kandidat tidak boleh sama dengan data yang sudah ada
df_kandidat = df_kandidat[
    ~df_kandidat["Komentar_preprocessed"]
    .astype(str)
    .isin(preprocessed_digunakan)].copy()

# Kandidat juga tidak boleh duplicate satu sama lain
df_kandidat = df_kandidat[
    ~df_kandidat["Komentar_preprocessed"]
    .duplicated(keep=False)].copy()

print("Jumlah kandidat yang lolos:", len(df_kandidat))
print("\nDistribusi kandidat:")
print(df_kandidat["Kode Label"].value_counts().sort_index())

Jumlah kandidat yang lolos: 4383

Distribusi kandidat:
Kode Label
C0    1456
C1    2242
C2     685
Name: count, dtype: int64


In [ ]:
# 16.6 — MEMILIH KANDIDAT REPLACEMENT

replacement_log = []
kandidat_tersedia = df_kandidat.copy()
for idx_lama in index_replace:
    label_lama = df.loc[idx_lama, "Kode Label"]

    # Kandidat harus mempunyai label yang sama
    kandidat = kandidat_tersedia[
        kandidat_tersedia["Kode Label"]
        == label_lama].copy()

    if kandidat.empty:
        raise ValueError(
            f"Tidak ada kandidat pengganti "
            f"untuk {label_lama}, "
            f"index {idx_lama}")

    # Ambil kandidat pertama
    kandidat_row = kandidat.iloc[0]

    # Simpan log
    replacement_log.append({
        "Index Lama":idx_lama,
        "No Lama":df.loc[idx_lama, "No"],
        "Label":label_lama,
        "Komentar Lama":df.loc[idx_lama, "Komentar"],
        "Preprocessed Lama":df.loc[ idx_lama, "Komentar_preprocessed"],
        "No Pengganti":kandidat_row["No"],
        "Komentar Pengganti":kandidat_row["Komentar"],
        "Preprocessed Pengganti":kandidat_row["Komentar_preprocessed"]})

    # Hapus kandidat agar tidak dipakai lagi
    kandidat_tersedia = kandidat_tersedia.drop(index=kandidat_row.name)

df_replacement = pd.DataFrame(replacement_log)
print("Jumlah kandidat replacement:", len(df_replacement))
display(df_replacement)

Jumlah kandidat replacement: 6


,Index Lama,No Lama,Label,Komentar Lama,Preprocessed Lama,No Pengganti,Komentar Pengganti,Preprocessed Pengganti
0,523,1620.0,C2,INI ORANG DARI PARTAI SILIT IRENG SEJATINYA AN...,ini orang dari partai silit ireng sejatinya an...,7,Lebih mantab dan enak perdebatan Roy s vs rism...,lebih mantab dan enak perdebatan roy si vs ris...
1,61,1798.0,C0,AAMIIN,amin,8,Ada baiknya Bp Presiden berkenan memanggil Ibu...,ada baiknya bp presiden berkenan memanggil ibu...
2,271,7187.0,C0,Seeetuuuujuuuu,setuju,9,Kalo pak jokowi bikin partai pasti akan banyak...,kalau pak jokowi bikin partai pasti akan banya...
3,2853,6800.0,C0,Setujuuuu,setuju,17,jika dipikir pikir sistem pendidikan di Singap...,jika dipikir pikir sistem pendidikan di singap...
4,1695,3434.0,C2,@himadunoim2529 Astaga! 23 Warga Koja Keracu...,astaga! 23 warga koja keracunan usai santap na...,10,Nggak percaya sama Raja Panci Gongor kentut by...,tidak percaya sama raja panci gongor kentut ba...
5,1740,1557.0,C0,@antipki5178 👍👍👍👍👍,👍,31,@indrideinkenzie308 adik nya yang mana yah k...,adik nya yang mana ya kalau boleh tahu


In [ ]:
# 16.7 — APPLY 6 REPLACEMENT

for _, row in df_replacement.iterrows():
    idx = int(row["Index Lama"])
    kandidat = df_kandidat[df_kandidat["No"] == row["No Pengganti"]]

    if kandidat.empty:
        raise ValueError(
            f"Data pengganti No {row['No Pengganti']} "
            f"tidak ditemukan di df_kandidat.")

    kandidat = kandidat.iloc[0]
    df.loc[idx, "No"] = kandidat["No"]
    df.loc[idx, "Komentar"] = kandidat["Komentar"]
    df.loc[idx, "Label Awal"] = kandidat["Label Awal"]
    df.loc[idx, "Label Pakar"] = kandidat["Label Pakar"]
    df.loc[idx, "Pasal Dominan"] = kandidat["Pasal Dominan"]
    df.loc[idx, "Catatan"] = kandidat["Catatan"]
    df.loc[idx, "Kode Label"] = kandidat["Kode Label"]

    # Hasil preprocessing kandidat
    df.loc[idx, "Komentar_clean"] = (kandidat["Komentar_clean"])
    df.loc[idx, "Komentar_preprocessed"] = (kandidat["Komentar_preprocessed"])

print("6 data C0-C2 berhasil di-replace.")

6 data C0-C2 berhasil di-replace.


In [ ]:
# 16.8 — VALIDASI HASIL REPLACEMENT

print("Jumlah data:", len(df))
print("\nDistribusi label:")
print(df["Kode Label"].value_counts().sort_index())

df_duplikat_after = df[
    df["Komentar_preprocessed"]
    .duplicated(keep=False)].copy()

print("\nJumlah baris dalam kelompok duplicate:", len(df_duplikat_after))
display(df_duplikat_after[["No", "Komentar", "Komentar_preprocessed", "Kode Label" ]].sort_values("Komentar_preprocessed"))

Jumlah data: 3815

Distribusi label:
Kode Label
C0    763
C1    763
C2    763
C3    763
C4    763
Name: count, dtype: int64

Jumlah baris dalam kelompok duplicate: 6


,No,Komentar,Komentar_preprocessed,Kode Label
42,7211.0,@sobatnu7078 INI ORANG DARI PARTAI SILIT IRE...,ini orang dari partai silit ireng sejatinya an...,C3
2120,7709.0,@AchmadSyaikhu INI ORANG DARI PARTAI SILIT I...,ini orang dari partai silit ireng sejatinya an...,C3
424,4959.0,"Lanjut atas, \nGusDur sendiri saat jadi presid...","lanjut atas, gusdur sendiri saat jadi presiden...",C3
2653,6357.0,"Lanjut atas,\nGusDur sendiri saat jadi preside...","lanjut atas, gusdur sendiri saat jadi presiden...",C3
3254,3712.0,😊😊😊😊😊,😊,C4
3759,NaN,😊😊😊😊😊😊😊😊,😊,C4


In [ ]:
# 16.9 — SIMPAN DATASET HASIL REPLACEMENT

output_file = "dataset_preprocessed_C0-C4_revised.csv"
df.to_csv(output_file, index=False)
print(f"Dataset berhasil disimpan: {output_file}")

Dataset berhasil disimpan: dataset_preprocessed_C0-C4_revised.csv


In [ ]:
from google.colab import files
files.download("dataset_preprocessed_C0-C4_revised.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

17 — Simpan Dataset Preprocessed

In [ ]:
output_file = "/content/drive/MyDrive/Bimbingan_27-08/dataset_preprocessed_C0-C4_revised.csv"
df.to_csv(output_file,index=False)
print("Dataset berhasil disimpan")

Dataset berhasil disimpan


18 — Preview Dataset

In [ ]:
df[["Komentar_preprocessed","Kode Label"]].head()

,Komentar_preprocessed,Kode Label
0,apakah seorang maling anaknya juga akan menjad...,C4
1,"​-wx1cciyaa betul, meni hese ngesahkeun pearam...",C2
2,omongan nya pak bahlil sering kontradiksi. lah...,C0
3,astagfirullah. katanya wakil rakyat tetapi kel...,C2
4,"cobalah pampang ke45 poin itu di media sosial,...",C4
